In [ ]:
# ============================================================
# Race Distribution Visualization
# ============================================================
# Purpose: Create a bar chart showing the distribution of race
#          (raracem column) in the HRS demographics table
# ============================================================

from pyspark.sql import functions as F
import matplotlib.pyplot as plt
import pandas as pd

# ============================================================
# DATA RETRIEVAL
# ============================================================

# Table details
catalog_name = "staging_catalog"
schema_name = "slv_cdm_hrs"
table_name = "hrs_demographics"
full_table_name = f"{catalog_name}.{schema_name}.{table_name}"

print(f"Reading data from: {full_table_name}")
print("="*60)

# Read the table
df = spark.table(full_table_name)

# ============================================================
# CALCULATE DISTRIBUTION
# ============================================================

# Group by race and count
race_distribution = df.groupBy("raracem").agg(
    F.count("*").alias("Count")
).orderBy("raracem")

# Convert to pandas for visualization
race_df = race_distribution.toPandas()

# Map numeric codes to descriptive labels
race_labels = {
    1.0: "White/Caucasian",
    2.0: "Black/African American",
    3.0: "Other"
}

# Apply labels, keeping null as "Not Specified"
race_df['race_label'] = race_df['raracem'].apply(
    lambda x: race_labels.get(x, "Not Specified") if pd.notna(x) else "Not Specified"
)

print("\nRace Distribution:")
display(race_df[['race_label', 'Count']])

# ============================================================
# CREATE BAR CHART
# ============================================================

# Create figure and axis
fig, ax = plt.subplots(figsize=(10, 6))

# Create bar chart with descriptive labels
colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']
ax.bar(race_df['race_label'], race_df['Count'], color=colors[:len(race_df)])

# Customize chart
ax.set_xlabel('Race Category', fontsize=12, fontweight='bold')
ax.set_ylabel('Count', fontsize=12, fontweight='bold')
ax.set_title('Distribution of Race Categories in HRS Demographics', fontsize=14, fontweight='bold', pad=20)

# Add value labels on top of bars
for i, v in enumerate(race_df['Count']):
    ax.text(i, v + max(race_df['Count'])*0.01, f'{v:,}', 
            ha='center', va='bottom', fontsize=10, fontweight='bold')

# Add grid for better readability
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_axisbelow(True)

# Adjust layout
plt.tight_layout()

# Display the chart
plt.show()

print("\n" + "="*60)
print(f"Chart Complete: Total records = {race_df['Count'].sum():,}")
print("="*60)